In [1]:
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import torch.optim as optim
import copy
from torch.utils.data import DataLoader
from torch.utils.data import random_split # use for data distribution for clients

import sys
import os
sys.path.append(os.path.abspath("../.."))

from client_training import train_local
from model_averaging import aggregate
from acc_evaluation import evaluate_model
from src.CNN_implementation import CNN

In [2]:
# Data Loading
transform = transforms.ToTensor()
mnist_trainset=datasets.MNIST(root = './data', train = True, download = True, transform = transform)
mnist_testset=datasets.MNIST(root = './data', train = False, download = True, transform = transform)
image, label = mnist_trainset[0]

In [3]:
# CNN
# a CNN with 2 layers: Layer1-> Relu->Pool->Layer2->...->Flatten->Fully connected->Output

global_model = CNN()
out = global_model(image.unsqueeze(0))
print(out.shape)

torch.Size([1, 10])


In [4]:
# Client abstraction, split dataset into clients
number_of_clients = 5

dataset_size = len(mnist_trainset)
client_size = dataset_size // number_of_clients # we use // instead of / to remove the fractional part
client_datasets = random_split(mnist_trainset, [client_size]*number_of_clients) # a list with: client_datasets[0], client_datasets[1], ...[5]

client_loaders = []
for dataset in client_datasets:
    loader = DataLoader(dataset, batch_size = 64, shuffle = True)
    client_loaders.append(loader)

# print(len(client_loaders))
# print(len(client_loaders[0]))
# images, labels = next(iter(client_loaders[0]))
# print(images.shape)

In [5]:
# Now we need to aggregate the results with FedAvg
number_of_rounds = 3
for round in range(number_of_rounds):
    print("---------------------------------")
    print(f"Round {round+1}:\n")
    
    # weights = train_local(local_model, client_loaders[0], epochs = 3, lr = 0.1)
    client_state_dictionary = []
    number_of_samples = []
    # loop over all clients
    for client in range(number_of_clients):
        print(f"Client{client+1}:")
        local_model = copy.deepcopy(global_model) # copy global model locally
        weights = train_local(local_model, client_loaders[client], epochs = 1, lr = 0.1) # train it
        client_state_dictionary.append(weights) # store the weights
        number_of_samples.append(len(client_loaders[client].dataset)) # store number of samples
        
    # aggregate
    global_model = aggregate(number_of_samples, client_state_dictionary, global_model)
    
    # compute accuracy
    test_loader = DataLoader(mnist_testset, batch_size=64, shuffle=False)
    evaluate_model(test_loader, global_model)


---------------------------------
Round 1:

Client1:
Epoch 1/1, Average Loss: 0.7707

Client2:
Epoch 1/1, Average Loss: 0.7757

Client3:
Epoch 1/1, Average Loss: 0.7784

Client4:
Epoch 1/1, Average Loss: 0.7735

Client5:
Epoch 1/1, Average Loss: 0.7824

Accuracy 94.47%

---------------------------------
Round 2:

Client1:
Epoch 1/1, Average Loss: 0.1932

Client2:
Epoch 1/1, Average Loss: 0.1941

Client3:
Epoch 1/1, Average Loss: 0.2027

Client4:
Epoch 1/1, Average Loss: 0.1907

Client5:
Epoch 1/1, Average Loss: 0.2010

Accuracy 96.21%

---------------------------------
Round 3:

Client1:
Epoch 1/1, Average Loss: 0.1414

Client2:
Epoch 1/1, Average Loss: 0.1395

Client3:
Epoch 1/1, Average Loss: 0.1542

Client4:
Epoch 1/1, Average Loss: 0.1362

Client5:
Epoch 1/1, Average Loss: 0.1430

Accuracy 97.08%

